# Analysis Pipeline — FGVC Aircraft / SHViT vs Baselines

Runs the full analysis after all 6 models are trained. Everything stays on
Colab's local SSD (`/content/`) for fast iteration; final artifacts are
mirrored into `<OUT_ROOT>/Stage 4: Benchmarking and Demo/analysis/` (and
optionally to Drive) so the project root layout matches the rest of the
experiment.

Default `OUT_ROOT` is `/content/CV_Research_Paper_FGVCAircraft` (or
`/content/drive/MyDrive/CV_Research_Paper_FGVCAircraft` when Drive is mounted).


## 0. GPU check

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

## 1. Mount Drive (read-only access to checkpoints + dataset source)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Source paths on Drive (where Stage 2 + Stage 3 saved checkpoints and dataset)
DRIVE_OUT_ROOT      = '/content/drive/MyDrive/CV_Research_Paper_FGVCAircraft'
DRIVE_DATA          = '/content/drive/MyDrive/fgvc_aircraft_data'

# Mirror the same layout locally under CV_Research_Paper_FGVCAircraft/.
OUT_ROOT            = '/content/CV_Research_Paper_FGVCAircraft'
ANALYSIS_ROOT       = f'{OUT_ROOT}/Stage 4: Benchmarking and Demo/analysis'

LOCAL_DATA          = '/content/fgvc_aircraft_local'  # large(-ish), kept separate
LOCAL_CHECKPOINTS   = OUT_ROOT
LOCAL_RESULTS       = f'{ANALYSIS_ROOT}/results'
LOCAL_FIGURES       = f'{ANALYSIS_ROOT}/figures'
LOCAL_EDA           = f'{ANALYSIS_ROOT}/eda_outputs'
LOCAL_ERROR         = f'{ANALYSIS_ROOT}/error_analysis_outputs'
LOCAL_DEMO          = f'{ANALYSIS_ROOT}/demo_output'

import os
for p in [ANALYSIS_ROOT, LOCAL_RESULTS, LOCAL_FIGURES,
          LOCAL_EDA, LOCAL_ERROR, LOCAL_DEMO]:
    os.makedirs(p, exist_ok=True)

print('Output root        :', OUT_ROOT)
print('Local analysis root:', ANALYSIS_ROOT)


## 2. Copy dataset to local SSD

If `/content/fgvc_aircraft_local` already exists from an earlier session, this skips the copy.


In [ ]:
import shutil, time

if not os.path.exists(LOCAL_DATA):
    if os.path.isdir(DRIVE_DATA):
        print('Copying FGVC Aircraft from Drive to local SSD...')
        t0 = time.time()
        shutil.copytree(DRIVE_DATA, LOCAL_DATA)
        print(f'Done in {time.time()-t0:.0f}s')
    else:
        # No cached copy in Drive — let prepare_fgvc_aircraft.py download it locally.
        os.makedirs(LOCAL_DATA, exist_ok=True)
else:
    img_root = os.path.join(LOCAL_DATA, 'fgvc_aircraft', 'jpg')
    if os.path.isdir(img_root):
        classes = len(os.listdir(img_root))
        print(f'Already copied — {classes} class folders present')


## 3. Copy trained checkpoints from Drive to the local output root

Mirrors the `<OUT_ROOT>/Stage 2|3/<model>/best.pth` layout that
`evaluate_all.py` and `make_figures.py` expect.


In [ ]:
# Layout matches what evaluate_all.py expects:
#   <OUT_ROOT>/Stage 2: baseline models/<model>/best.pth
#   <OUT_ROOT>/Stage 3: fine-tuning SHViT/<model>/best.pth
STAGE2_LOCAL = f'{LOCAL_CHECKPOINTS}/Stage 2: baseline models'
STAGE3_LOCAL = f'{LOCAL_CHECKPOINTS}/Stage 3: fine-tuning SHViT'
os.makedirs(STAGE2_LOCAL, exist_ok=True)
os.makedirs(STAGE3_LOCAL, exist_ok=True)

DRIVE_STAGE2 = f'{DRIVE_OUT_ROOT}/Stage 2: baseline models'
DRIVE_STAGE3 = f'{DRIVE_OUT_ROOT}/Stage 3: fine-tuning SHViT'

for model in ['resnet50', 'mobilenet_v2']:
    src_dir = f'{DRIVE_STAGE2}/{model}'
    dst_dir = f'{STAGE2_LOCAL}/{model}'
    if os.path.isdir(src_dir) and not os.path.isdir(dst_dir):
        shutil.copytree(src_dir, dst_dir)
        print(f'copied {model}: {os.path.getsize(f"{dst_dir}/best.pth")/1e6:.1f} MB')
    elif os.path.isdir(dst_dir):
        print(f'{model}: already present')
    else:
        print(f'WARNING: {model} not found at {src_dir}')

for model in ['shvit_s1', 'shvit_s2', 'shvit_s3', 'shvit_s4']:
    src_dir = f'{DRIVE_STAGE3}/{model}'
    dst_dir = f'{STAGE3_LOCAL}/{model}'
    if os.path.isdir(src_dir) and not os.path.isdir(dst_dir):
        shutil.copytree(src_dir, dst_dir)
        print(f'copied {model}: {os.path.getsize(f"{dst_dir}/best.pth")/1e6:.1f} MB')
    elif os.path.isdir(dst_dir):
        print(f'{model}: already present')
    else:
        print(f'WARNING: {model} not found at {src_dir}')


## 4. Clone repos and install dependencies

Pulls the latest analysis scripts from your branch. If the repo already exists, `git pull` updates it.

In [ ]:
REPO_DIR  = '/content/Vision_Project_spring_26'
REPO_URL  = 'https://github.com/saif-farid-tech/Vision_Project_spring_26.git'
BRANCH    = 'Vision_Project_spring_26_FGVCAircraft'
SHVIT_DIR = '/content/SHViT'

import os, shutil

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

if not os.path.isdir(SHVIT_DIR):
    !git clone https://github.com/ysj9909/SHViT.git {SHVIT_DIR}

# Copy analysis scripts to /content for clean invocation
for fname in ['eda.py', 'evaluate_all.py', 'make_figures.py',
              'error_analysis.py', 'demo.py',
              'augmentation.py', 'metrics.py', 'splits.py',
              'prepare_fgvc_aircraft.py']:
    s = f'{REPO_DIR}/{fname}'
    if os.path.exists(s):
        shutil.copy(s, f'/content/{fname}')
        print(f'copied {fname}')
    else:
        print(f'MISSING: {fname} (push it to your branch first)')

DATASETS_DST = '/content/datasets'
if os.path.isdir(DATASETS_DST):
    shutil.rmtree(DATASETS_DST)
shutil.copytree(f'{REPO_DIR}/datasets', DATASETS_DST)
print('Vendored datasets/ package.')


In [ ]:
# Dependencies: timm pinned, fvcore for GFLOPs, seaborn for heatmaps.
# FGVC Aircraft uses its own canonical split files, so no gdown needed.
!pip install -q timm==0.5.4 --no-deps
!pip install -q einops==0.4.1 easydict fvcore seaborn
print('Deps installed.')


## 5. EDA — sample grid, class distribution, image sizes

Quick (~2 min). No GPU needed.

In [ ]:
!python /content/eda.py \
    --data-root  {LOCAL_DATA} \
    --output-dir "{LOCAL_EDA}"

print('\nEDA outputs:')
for f in sorted(os.listdir(LOCAL_EDA)):
    sz = os.path.getsize(f'{LOCAL_EDA}/{f}') / 1024
    print(f'  {f}  ({sz:.0f} KB)')


## 6. Full evaluation + benchmarks on the test set

Runs all 6 models on the FGVC Aircraft test split (the canonical
`images_variant_test.txt`), plus parameter / FLOPs / throughput / latency
benchmarks.


In [ ]:
!python /content/evaluate_all.py \
    --checkpoints-dir "{LOCAL_CHECKPOINTS}" \
    --shvit-dir       {SHVIT_DIR} \
    --data-root       {LOCAL_DATA} \
    --output-dir      "{LOCAL_RESULTS}"

print('\nResults outputs:')
for root, dirs, files in os.walk(LOCAL_RESULTS):
    for f in files:
        full = os.path.join(root, f)
        rel  = os.path.relpath(full, LOCAL_RESULTS)
        sz   = os.path.getsize(full) / 1024
        print(f'  {rel}  ({sz:.0f} KB)')


In [ ]:
# Quick peek at the headline table
table_path = f'{LOCAL_RESULTS}/results_table.md'
if os.path.exists(table_path):
    with open(table_path) as f:
        print(f.read())

## 7. Generate report figures

Reads training logs (from the checkpoint dirs) + `results.json` files. Produces the speed-vs-accuracy headline figure, training curves, train loss, accuracy bars.

In [ ]:
!python /content/make_figures.py \
    --results-dir "{LOCAL_RESULTS}" \
    --logs-dir    "{LOCAL_CHECKPOINTS}" \
    --output-dir  "{LOCAL_FIGURES}"

print('\nFigures:')
for f in sorted(os.listdir(LOCAL_FIGURES)):
    sz = os.path.getsize(f'{LOCAL_FIGURES}/{f}') / 1024
    print(f'  {f}  ({sz:.0f} KB)')


In [ ]:
# Inline preview
from IPython.display import Image, display
for f in ['fig_speed_vs_accuracy.png', 'fig_training_curves.png',
          'fig_accuracy_bars.png', 'fig_train_loss.png']:
    p = f'{LOCAL_FIGURES}/{f}'
    if os.path.exists(p):
        print(f)
        display(Image(p, width=700))

## 8. Error analysis on SHViT-S4

Worst-15 categories, top-10 confused class pairs, confusion heatmap, misclassified image grid.

In [ ]:
S4_CKPT = f'{STAGE3_LOCAL}/shvit_s4/best.pth'

!python /content/error_analysis.py \
    --results-dir "{LOCAL_RESULTS}" \
    --data-root   {LOCAL_DATA} \
    --shvit-dir   {SHVIT_DIR} \
    --checkpoint  "{S4_CKPT}" \
    --output-dir  "{LOCAL_ERROR}"

print('\nError analysis outputs:')
for f in sorted(os.listdir(LOCAL_ERROR)):
    sz = os.path.getsize(f'{LOCAL_ERROR}/{f}') / 1024
    print(f'  {f}  ({sz:.0f} KB)')


In [ ]:
# Inline preview of error figures + summary text
for f in ['confusion_top20.png', 'misclassified_grid.png', 'worst_15_categories.png']:
    p = f'{LOCAL_ERROR}/{f}'
    if os.path.exists(p):
        print(f)
        display(Image(p, width=700))

summary = f'{LOCAL_ERROR}/error_analysis_summary.txt'
if os.path.exists(summary):
    with open(summary) as f:
        print(f.read())

## 9. Demo on a single image

Smoke test for `demo.py`. By default this grabs the first image from
`fgvc_aircraft/images/`; edit `DEMO_IMAGE` to try any other aircraft image.


In [ ]:
import glob
# FGVC Aircraft ships all images in a single flat images/ folder.
candidates = sorted(glob.glob(f'{LOCAL_DATA}/fgvc_aircraft/images/*.jpg'))
DEMO_IMAGE = candidates[0]
print('Using image:', DEMO_IMAGE)

DEMO_OUT = f'{LOCAL_DEMO}/demo_output.png'

!python /content/demo.py \
    --image       "{DEMO_IMAGE}" \
    --checkpoint  "{S4_CKPT}" \
    --shvit-dir   {SHVIT_DIR} \
    --data-root   {LOCAL_DATA} \
    --output      "{DEMO_OUT}"

if os.path.exists(DEMO_OUT):
    display(Image(DEMO_OUT, width=600))


## 10. Bundle everything for download

Zips the entire `<OUT_ROOT>/Stage 4: Benchmarking and Demo/analysis/`
folder so you can download it from the Colab file browser
(right-click `analysis.zip` → Download).


In [ ]:
ZIP_PATH = '/content/CV_Research_Paper_FGVCAircraft_analysis.zip'
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# Zip the analysis subtree (results, figures, EDA, error analysis, demo)
!cd "{ANALYSIS_ROOT}" && zip -rq {ZIP_PATH} \
    results figures eda_outputs error_analysis_outputs demo_output

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'Bundle ready: {ZIP_PATH}  ({size_mb:.1f} MB)')
print('Right-click in the file browser to download.')


In [ ]:
# Optional: also copy the bundle to Drive for safety
DRIVE_BUNDLE = '/content/drive/MyDrive/CV_Research_Paper_FGVCAircraft_analysis.zip'
shutil.copy(ZIP_PATH, DRIVE_BUNDLE)
print(f'Also saved to: {DRIVE_BUNDLE}')
